# Heritage Short: ACE-Step 1.5 score candidates

Generates `candidates` instrumental pieces (one per seed) from the storyboard's music cues, a little longer than the film. The pipeline trims the chosen one to length with a fade.

**Before running**
1. Locally: `py -m pipeline music --export outputs/kaggle/music_job`, then upload that folder as a **Kaggle dataset** and attach it here.
2. Settings: **GPU** accelerator, **Internet on**.

**Outputs:** `/kaggle/working/out/score_<hash>_c<N>.wav` (don't rename them) and `runs_fragment.jsonl`.
Download `out.zip`, then locally: `py -m pipeline music --import <folder>`. Listen to the candidates *under the narration*, set `score.chosen` in `storyboard.json`, and write down why in EXPERIMENTS.md.

In [ ]:
import json, time, datetime, subprocess, shutil, glob
from pathlib import Path

RUN_ID = "kaggle-" + datetime.datetime.utcnow().strftime("%Y%m%d-%H%M%S")
OUT = Path("/kaggle/working/out"); OUT.mkdir(parents=True, exist_ok=True)
FRAGMENT = OUT / "runs_fragment.jsonl"

def log(stage, shot, model, params, seed, status, latency, output=None, error=None):
    """One line in the pipeline's runs.jsonl format; `--import` merges it locally."""
    rec = {"ts": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"), "run_id": RUN_ID,
           "stage": stage, "shot": shot, "provider": "kaggle", "model": model, "params": params,
           "seed": seed, "input_hash": None, "status": status, "attempt": 1,
           "latency_s": round(latency, 3), "gpu_s": round(latency, 3), "cost_usd_list": None,
           "output": output, "error": (error or None) and str(error)[:500]}
    with open(FRAGMENT, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def find_job():
    hits = glob.glob("/kaggle/input/**/job.json", recursive=True)
    assert hits, "attach the exported job folder as a Kaggle dataset (it must contain job.json)"
    path = Path(hits[0]); return path.parent, json.loads(path.read_text())

JOB_DIR, JOB = find_job()
print(json.dumps({k: JOB[k] for k in ("caption", "duration_s", "bpm", "key", "seeds")}, indent=1))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/ACE-Step/ACE-Step-1.5.git /kaggle/working/ACE-Step-1.5
!cd /kaggle/working/ACE-Step-1.5 && pip install -q -e .

**Read the interface before generating.** This repository's usage was not verified from the development machine. The next cell prints the README's usage section.
Adjust **only** the body of `generate()` (marked ADAPT) to match it. Keep: instrumental, the caption, the duration, bpm/key in their own fields (not in the caption), one seed per candidate, and a WAV at `out_path`.

In [ ]:
import re
readme = Path("/kaggle/working/ACE-Step-1.5/README.md").read_text(errors="ignore")
for block in re.findall(r"```(?:bash|shell|python|sh)?\n(.*?)```", readme, flags=re.S)[:12]:
    print(block.strip()[:600]); print("-" * 60)

In [ ]:
def generate(caption, duration_s, bpm, key, seed, out_path):
    """ADAPT to the README above if needed. Must write a WAV file to out_path."""
    cmd = ["python", "-m", "acestep.infer",            # <- entry point: check the README
           "--caption", caption, "--duration", str(duration_s), "--bpm", str(bpm),
           "--key", str(key), "--instrumental", "--seed", str(seed), "--output", str(out_path)]
    r = subprocess.run(cmd, cwd="/kaggle/working/ACE-Step-1.5", capture_output=True, text=True)
    if r.returncode != 0 or not Path(out_path).exists():
        raise RuntimeError(r.stderr[-1500:])

params = {k: JOB[k] for k in ("duration_s", "bpm", "key", "caption")}
for seed, name in zip(JOB["seeds"], JOB["outs"]):
    start = time.time()
    try:
        generate(JOB["caption"], JOB["duration_s"], JOB["bpm"], JOB["key"], seed, OUT / name)
        log("music", f"c{seed}", JOB["model"], params, seed, "ok", time.time() - start, name)
        print(name, "ok", f"{time.time()-start:.0f} s")
    except Exception as e:
        log("music", f"c{seed}", JOB["model"], params, seed, "error", time.time() - start, None, e)
        print(name, "FAILED:", str(e)[-800:])

In [ ]:
shutil.make_archive("/kaggle/working/out", "zip", OUT)
print(sorted(p.name for p in OUT.iterdir()))
print("Download /kaggle/working/out.zip (Output tab), unzip, then: py -m pipeline music --import <folder>")